# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
import joblib


## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df_features = pd.read_csv('../../src/data/day-of-week-not-scaled.csv')


In [3]:
df_target = pd.read_csv('../../src/data/dayofweek.csv')
y = df_target['dayofweek']

X_train, X_test, y_train, y_test = train_test_split(
    df_features, y, test_size=0.2, random_state=21, stratify=y
)


In [4]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, random_state=21, stratify=y_train
)


## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [5]:
svm = SVC(C=10, gamma='auto', kernel='rbf', random_state=21, probability=True)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_valid)
print(f'accuracy is {accuracy_score(y_valid, y_pred):.5f}')
print(f'precision is {precision_score(y_valid, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')


accuracy is 0.87778
precision is 0.88162
recall is 0.87778


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [6]:
dt = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=23, random_state=21)
dt.fit(X_train, y_train)
y_pred = dt.predict(X_valid)
print(f'accuracy is {accuracy_score(y_valid, y_pred):.5f}')
print(f'precision is {precision_score(y_valid, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')


accuracy is 0.86667
precision is 0.86975
recall is 0.86667


In [7]:
rf = RandomForestClassifier(n_estimators=50, max_depth=28, random_state=21)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_valid)
print(f'accuracy is {accuracy_score(y_valid, y_pred):.5f}')
print(f'precision is {precision_score(y_valid, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')


accuracy is 0.88889
precision is 0.88952
recall is 0.88889


## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [8]:
vh = VotingClassifier(estimators=[
    ('svm', svm), ('dt', dt), ('rf', rf)
], voting='hard')
vh.fit(X_train, y_train)
y_pred = vh.predict(X_valid)
print(f'Hard voting:')
print(f'accuracy is {accuracy_score(y_valid, y_pred):.5f}')
print(f'precision is {precision_score(y_valid, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Hard voting:
accuracy is 0.89259
precision is 0.89236
recall is 0.89259


In [9]:
vs = VotingClassifier(estimators=[
    ('svm', svm), ('dt', dt), ('rf', rf)
], voting='soft')
vs.fit(X_train, y_train)
y_pred = vs.predict(X_valid)
print(f'Soft voting:')
print(f'accuracy is {accuracy_score(y_valid, y_pred):.5f}')
print(f'precision is {precision_score(y_valid, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Soft voting:
accuracy is 0.88519
precision is 0.88816
recall is 0.88519


In [10]:
best_voting = None
best_score = 0
best_precision = 0

for w1 in range(1, 6):
    for w2 in range(1, 6):
        for w3 in range(1, 6):
            vc = VotingClassifier(estimators=[
                ('svm', svm), ('dt', dt), ('rf', rf)
            ], voting='soft', weights=[w1, w2, w3])
            vc.fit(X_train, y_train)
            y_pred = vc.predict(X_valid)
            acc = accuracy_score(y_valid, y_pred)
            prec = precision_score(y_valid, y_pred, average='weighted')
            if acc > best_score or (acc == best_score and prec > best_precision):
                best_score = acc
                best_precision = prec
                best_voting = (w1, w2, w3, vc)
                best_pred = y_pred

w1, w2, w3, vc = best_voting
print(f'Best weights found: [{w1}, {w2}, {w3}]')
print(f'accuracy is {best_score:.5f}')
print(f'precision is {best_precision:.5f}')
print(f'recall is {recall_score(y_valid, best_pred, average="weighted"):.5f}')

print()
print('Weights [4,1,4]:')
vc414 = VotingClassifier(estimators=[
    ('svm', svm), ('dt', dt), ('rf', rf)
], voting='soft', weights=[4, 1, 4])
vc414.fit(X_train, y_train)
y_pred = vc414.predict(X_valid)
print(f'accuracy is {accuracy_score(y_valid, y_pred):.5f}')
print(f'precision is {precision_score(y_valid, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

Best weights found: [5, 1, 2]
accuracy is 0.91111
precision is 0.91627
recall is 0.91111

Weights [4,1,4]:
accuracy is 0.90370
precision is 0.90629
recall is 0.90370


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [11]:
voting_best = VotingClassifier(estimators=[
    ('svm', svm), ('dt', dt), ('rf', rf)
], voting='soft', weights=[4, 1, 4])
voting_best.fit(X_train, y_train)
y_pred = voting_best.predict(X_valid)
print(f'Voting [4,1,4] on valid:')
print(f'accuracy is {accuracy_score(y_valid, y_pred):.5f}')
print(f'precision is {precision_score(y_valid, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Voting [4,1,4] on valid:
accuracy is 0.90370
precision is 0.90629
recall is 0.90370


In [12]:
voting_best.fit(X_train, y_train)
y_pred = voting_best.predict(X_test)
print(f'Voting [4,1,4] on test:')
print(f'accuracy is {accuracy_score(y_test, y_pred):.5f}')
print(f'precision is {precision_score(y_test, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_test, y_pred, average="weighted"):.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Voting [4,1,4] on test:
accuracy is 0.89941
precision is 0.90310
recall is 0.89941


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [13]:
svm_base = SVC(C=10, gamma='auto', kernel='rbf', random_state=21, probability=True)

best_bagging = None
best_score = 0
best_precision = 0

for n in [10, 20, 30, 40, 50, 60]:
    bc = BaggingClassifier(estimator=svm_base, n_estimators=n, random_state=21)
    bc.fit(X_train, y_train)
    y_pred = bc.predict(X_valid)
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, average='weighted')
    print(f'n_estimators={n:2d}: accuracy={acc:.5f}, precision={prec:.5f}')
    if acc > best_score or (acc == best_score and prec > best_precision):
        best_score = acc
        best_precision = prec
        best_bagging = (n, bc, y_pred)

n, bc, y_pred = best_bagging
print(f'\nBest n_estimators = {n}')
print(f'accuracy is {best_score:.5f}')
print(f'precision is {best_precision:.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

n_estimators=10: accuracy=0.88519, precision=0.89427


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

n_estimators=20: accuracy=0.88519, precision=0.89258


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

...

n_estimators=40: accuracy=0.88148, precision=0.89111


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

n_estimators=50: accuracy=0.88148, precision=0.89035


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

n_estimators=60: accuracy=0.88148, precision=0.89035

Best n_estimators = 30
accuracy is 0.88889
precision is 0.89718
recall is 0.88889


In [14]:
bc.fit(X_train, y_train)
y_pred = bc.predict(X_test)
print(f'Bagging on test:')
print(f'accuracy is {accuracy_score(y_test, y_pred):.5f}')
print(f'precision is {precision_score(y_test, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_test, y_pred, average="weighted"):.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

Bagging on test:
accuracy is 0.87278
precision is 0.87840
recall is 0.87278


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [15]:
best_stacking = None
best_score = 0
best_precision = 0

for n_splits in [2, 3, 4, 5, 6, 7]:
    for passthrough in [True, False]:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        sc = StackingClassifier(
            estimators=[('svm', svm), ('dt', dt), ('rf', rf)],
            final_estimator=LogisticRegression(solver='lbfgs', max_iter=1000),
            cv=cv, passthrough=passthrough
        )
        sc.fit(X_train, y_train)
        y_pred = sc.predict(X_valid)
        acc = accuracy_score(y_valid, y_pred)
        prec = precision_score(y_valid, y_pred, average='weighted')
        print(f'n_splits={n_splits}, passthrough={passthrough}: accuracy={acc:.5f}, precision={prec:.5f}')
        if acc > best_score or (acc == best_score and prec > best_precision):
            best_score = acc
            best_precision = prec
            best_stacking = (n_splits, passthrough, sc, y_pred)

n_splits, passthrough, sc, y_pred = best_stacking
print(f'\nBest: n_splits={n_splits}, passthrough={passthrough}')
print(f'accuracy is {best_score:.5f}')
print(f'precision is {best_precision:.5f}')
print(f'recall is {recall_score(y_valid, y_pred, average="weighted"):.5f}')



/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


n_splits=2, passthrough=True: accuracy=0.89630, precision=0.89901


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

n_splits=2, passthrough=False: accuracy=0.89630, precision=0.89817


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

n_splits=6, passthrough=False: accuracy=0.90741, precision=0.90909


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

n_splits=7, passthrough=True: accuracy=0.90741, precision=0.91031


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

n_splits=7, passthrough=False: accuracy=0.90370, precision=0.90581

Best: n_splits=5, passthrough=False
accuracy is 0.92222
precision is 0.92464
recall is 0.92222


In [16]:
sc.fit(X_train, y_train)
y_pred = sc.predict(X_test)
print(f'Stacking on test:')
print(f'accuracy is {accuracy_score(y_test, y_pred):.5f}')
print(f'precision is {precision_score(y_test, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_test, y_pred, average="weighted"):.5f}')


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The

Stacking on test:
accuracy is 0.90828
precision is 0.90994
recall is 0.90828


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [17]:
best_model = VotingClassifier(estimators=[
    ('svm', svm), ('dt', dt), ('rf', rf)
], voting='soft', weights=[4, 1, 4])
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(f'Best model (Voting [4,1,4]) on test:')
print(f'accuracy is {accuracy_score(y_test, y_pred):.5f}')
print(f'precision is {precision_score(y_test, y_pred, average="weighted"):.5f}')
print(f'recall is {recall_score(y_test, y_pred, average="weighted"):.5f}')


Best model (Voting [4,1,4]) on test:
accuracy is 0.89941
precision is 0.90310
recall is 0.89941


/Users/leyla_iz/Desktop/21/cybernor/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [18]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
class_counts = y_test.value_counts().sort_index()

print('Errors by weekday:')
for i in class_counts.index:
    total = class_counts[i]
    errors = cm[i].sum() - cm[i, i]
    print(f'  Class {i}: {errors}/{total} = {100*errors/total:.2f}%')

worst_weekday = max(class_counts.index, key=lambda i: (cm[i].sum() - cm[i,i]) / class_counts[i] * 100)
print(f'\nMost errors for weekday {worst_weekday}')

errors_mask = y_pred != y_test
X_errors = X_test[errors_mask.values]
y_error_actual = y_test[errors_mask.values]

lab_cols = [c for c in df_features.columns if c.startswith('labname')]
print('\nErrors by labname:')
for col in lab_cols:
    error_count = X_errors[col].sum()
    total = X_test[col].sum()
    if total > 0:
        print(f'  {col}: {error_count:.0f}/{total:.0f} = {100*error_count/total:.2f}%')

worst_lab = max(lab_cols, key=lambda c: X_errors[c].sum() / max(X_test[c].sum(), 1) * 100 if X_test[c].sum() > 0 else 0)
print(f'\nMost errors for labname {worst_lab}')

user_cols = [c for c in df_features.columns if c.startswith('uid')]
print('\nErrors by user:')
for col in user_cols:
    error_count = X_errors[col].sum()
    total = X_test[col].sum()
    if total > 0:
        print(f'  {col}: {error_count:.0f}/{total:.0f} = {100*error_count/total:.2f}%')

worst_user = max(user_cols, key=lambda c: X_errors[c].sum() / max(X_test[c].sum(), 1) * 100 if X_test[c].sum() > 0 else 0)
print(f'\nMost errors for user {worst_user}')


Errors by weekday:
  Class 0: 8/27 = 29.63%
  Class 1: 5/55 = 9.09%
  Class 2: 2/30 = 6.67%
  Class 3: 4/80 = 5.00%
  Class 4: 2/21 = 9.52%
  Class 5: 7/54 = 12.96%
  Class 6: 6/71 = 8.45%

Most errors for weekday 0

Errors by labname:
  labname_code_rvw: 1/13 = 7.69%
  labname_lab03: 1/1 = 100.00%
  labname_lab03s: 0/1 = 0.00%
  labname_lab05s: 1/6 = 16.67%
  labname_laba04: 9/35 = 25.71%
  labname_laba04s: 6/25 = 24.00%
  labname_laba05: 0/47 = 0.00%
  labname_laba06: 1/9 = 11.11%
  labname_laba06s: 2/15 = 13.33%
  labname_project1: 13/186 = 6.99%

Most errors for labname labname_lab03

Errors by user:
  uid_user_1: 0/9 = 0.00%
  uid_user_10: 0/12 = 0.00%
  uid_user_12: 0/12 = 0.00%
  uid_user_13: 2/17 = 11.76%
  uid_user_14: 3/31 = 9.68%
  uid_user_15: 0/2 = 0.00%
  uid_user_16: 1/5 = 20.00%
  uid_user_17: 2/7 = 28.57%
  uid_user_18: 1/6 = 16.67%
  uid_user_19: 3/19 = 15.79%
  uid_user_2: 4/28 = 14.29%
  uid_user_20: 0/20 = 0.00%
  uid_user_21: 1/14 = 7.14%
  uid_user_22: 1/1 = 100.

In [19]:
joblib.dump(best_model, 'best_model_ex03.joblib')


['best_model_ex03.joblib']